# BD-TrafficGuard — Robust Bangladeshi Traffic Sign Detection & Recognition

This notebook implements the project requirements from the supplied proposal and presentation:

- Bangladesh Road Traffic Sign Dataset: detection with 31 classes
- clean train/validation/test split
- YOLO baseline
- degradation-aware robust YOLO
- controlled corruption benchmark: Gaussian blur, motion blur, low light, glare, fog, rain, occlusion, JPEG compression, small-object/resolution degradation
- clean vs corrupted evaluation
- mAP@0.5, mAP@0.5:0.95, precision, recall, F1
- robustness drop
- class-wise analysis
- inference time / FPS / model size
- ablation experiments
- qualitative failure analysis
- optional Faster R-CNN comparison
- optional two-stage detector + classifier using EfficientNet-B0
- export for lightweight deployment

**Important:** set `DATASET_SOURCE` in Cell 2 to your downloaded dataset location. The notebook is designed to work with a YOLO-format dataset or with a dataset that already contains image/annotation folders that can be converted.

In [ ]:
# ============================================================
# 1. INSTALL / IMPORTS
# ============================================================
# Run this cell once in a fresh environment.
# In Colab, restart the runtime only if the installer asks you to.

%pip -q install -U ultralytics opencv-python pillow pandas numpy matplotlib seaborn scikit-learn pyyaml tqdm torchmetrics torchvision

import os
import sys
import json
import time
import math
import random
import shutil
import zipfile
import warnings
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from ultralytics import YOLO

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

# CHANGE THIS:
# Point this to the downloaded Bangladesh Road Traffic Sign Dataset.
#
# Examples:
#   "/content/Bangladesh-Road-Traffic-Sign-Dataset"
#   "/content/dataset.zip"
#   "/kaggle/input/your-dataset-folder"
#
# If your dataset is already in YOLO format, the notebook will use it directly.
DATASET_SOURCE = "/content/bangladesh_traffic_sign_dataset"

PROJECT_ROOT = Path("/content/BD-TrafficGuard")
DATA_ROOT = PROJECT_ROOT / "data"
YOLO_DATASET = DATA_ROOT / "yolo_dataset"
RUNS_ROOT = PROJECT_ROOT / "runs"
CORRUPTION_ROOT = PROJECT_ROOT / "BD-TrafficSign-C"

# Main experimental settings
IMG_SIZE = 640
EPOCHS_BASELINE = 50
EPOCHS_ROBUST = 50
BATCH_SIZE = 16
WORKERS = 2

# Set to True for a quick smoke test before expensive training.
QUICK_TEST = False

if QUICK_TEST:
    EPOCHS_BASELINE = 2
    EPOCHS_ROBUST = 2
    BATCH_SIZE = 4

# Models requested in the proposal
YOLO_BASELINE = "yolo11n.pt"
YOLO_SMALL = "yolo11s.pt"

# Split requested by the proposal: 80/10/10
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
CORRUPTION_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset source:", DATASET_SOURCE)


In [ ]:
# ============================================================
# 3. DATASET INSPECTION
# ============================================================

source = Path(DATASET_SOURCE)

if not source.exists():
    print("DATASET_SOURCE does not exist yet.")
    print("Download/extract the Bangladesh Road Traffic Sign Dataset and update Cell 2.")
else:
    print("Dataset source found:", source)
    if source.is_file():
        print("File:", source.name, "size:", round(source.stat().st_size / (1024**2), 2), "MB")
    else:
        files = list(source.rglob("*"))
        image_files = [p for p in files if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}]
        txt_files = [p for p in files if p.suffix.lower() == ".txt"]
        print("Images:", len(image_files))
        print("TXT files:", len(txt_files))
        print("Top-level entries:", [p.name for p in list(source.iterdir())[:20]])


In [ ]:
# ============================================================
# 4. OPTIONAL ZIP EXTRACTION
# ============================================================

def extract_if_zip(source_path, destination):
    source_path = Path(source_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    if source_path.is_file() and source_path.suffix.lower() == ".zip":
        print("Extracting:", source_path)
        with zipfile.ZipFile(source_path, "r") as z:
            z.extractall(destination)
        return destination

    return source_path

if Path(DATASET_SOURCE).exists():
    RESOLVED_SOURCE = extract_if_zip(DATASET_SOURCE, DATA_ROOT / "raw_extracted")
else:
    RESOLVED_SOURCE = Path(DATASET_SOURCE)

print("Resolved source:", RESOLVED_SOURCE)


## Expected annotation format

The primary dataset described in the proposal contains JPG images and TXT annotations. The preferred YOLO label format is:

```text
class_id x_center y_center width height
```

where all coordinates are normalized to `[0, 1]`.

If your downloaded dataset already contains `train/images`, `train/labels`, `valid/images`, `valid/labels`, and `test/images`, the conversion step below will reuse it.

If the dataset uses another folder structure, update the small `find_*` helpers in Cell 5 rather than changing the rest of the notebook.

In [ ]:
# ============================================================
# 5. FIND / PREPARE YOLO DATASET
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_dir_case_insensitive(root, names):
    root = Path(root)
    names = {n.lower() for n in names}
    candidates = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in names]
    return candidates[0] if candidates else None

def find_split(root, split_names):
    for name in split_names:
        p = find_dir_case_insensitive(root, [name])
        if p:
            return p
    return None

def is_yolo_dataset(root):
    root = Path(root)
    yaml_files = list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))
    for y in yaml_files:
        try:
            data = yaml.safe_load(y.read_text())
            if isinstance(data, dict) and ("train" in data) and ("names" in data or "nc" in data):
                return y
        except Exception:
            pass

    train_img = find_dir_case_insensitive(root, ["train/images"])
    # The direct check above is intentionally conservative; use path patterns below.
    for p in root.rglob("images"):
        if p.parent.name.lower() in {"train", "valid", "val", "test"}:
            return None
    return None

def collect_image_label_pairs(root):
    root = Path(root)
    images = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    pairs = []

    for img in images:
        # Common YOLO layouts: images/foo.jpg -> labels/foo.txt
        candidates = [
            img.with_suffix(".txt"),
            img.parent.parent / "labels" / (img.stem + ".txt"),
            img.parent / "labels" / (img.stem + ".txt"),
        ]
        label = next((p for p in candidates if p.exists()), None)
        if label is not None:
            pairs.append((img, label))

    return pairs

pairs = collect_image_label_pairs(RESOLVED_SOURCE) if Path(RESOLVED_SOURCE).exists() else []
print("Image/label pairs found:", len(pairs))

if len(pairs) == 0:
    print("\nNo image/label pairs were automatically found.")
    print("Inspect the dataset folder structure above.")
    print("If annotations are already YOLO-format, place the dataset in:")
    print("  train/images, train/labels")
    print("  val/images, val/labels")
    print("  test/images, test/labels")


In [ ]:
# ============================================================
# 6. CLASS DISCOVERY FROM YOLO LABELS
# ============================================================

def discover_class_ids(label_files):
    ids = set()
    for lf in tqdm(label_files, desc="Reading labels"):
        try:
            for line in lf.read_text().splitlines():
                parts = line.strip().split()
                if not parts:
                    continue
                ids.add(int(float(parts[0])))
        except Exception:
            continue
    return sorted(ids)

if pairs:
    class_ids = discover_class_ids([l for _, l in pairs])
    print("Class IDs found:", class_ids)
    print("Number of classes:", len(class_ids))
    if class_ids != list(range(len(class_ids))):
        print("WARNING: class IDs are not contiguous from 0.")
else:
    class_ids = list(range(31))

# If the dataset provides class names in a YAML file, use them.
CLASS_NAMES = None
yaml_candidates = list(Path(RESOLVED_SOURCE).rglob("*.yaml")) + list(Path(RESOLVED_SOURCE).rglob("*.yml"))
for yf in yaml_candidates:
    try:
        d = yaml.safe_load(yf.read_text())
        if isinstance(d, dict) and "names" in d:
            names = d["names"]
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names, key=lambda x: int(x))]
            if len(names) >= len(class_ids):
                CLASS_NAMES = list(names)
                print("Class names loaded from:", yf)
                break
    except Exception:
        pass

if CLASS_NAMES is None:
    CLASS_NAMES = [f"class_{i}" for i in range(len(class_ids))]
    print("Class names were not found in a YAML file.")
    print("Replace CLASS_NAMES with the official 31 class names before final reporting.")

print("Classes:", CLASS_NAMES)


In [ ]:
# ============================================================
# 7. VISUALIZE RAW DATA + LABELS
# ============================================================

def read_yolo_labels(label_path):
    rows = []
    if not label_path.exists():
        return rows
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls, xc, yc, w, h = map(float, parts[:5])
            rows.append((int(cls), xc, yc, w, h))
    return rows

def draw_yolo_labels(img_path, label_path, class_names):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    for cls, xc, yc, w, h in read_yolo_labels(label_path):
        x1 = int((xc - w/2) * W)
        y1 = int((yc - h/2) * H)
        x2 = int((xc + w/2) * W)
        y2 = int((yc + h/2) * H)
        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
        name = class_names[cls] if cls < len(class_names) else str(cls)
        cv2.putText(img, name, (x1, max(20,y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,0,0), 2)
    return img

if pairs:
    sample_pairs = random.sample(pairs, min(9, len(pairs)))
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.ravel()
    for ax, (imgp, labp) in zip(axes, sample_pairs):
        vis = draw_yolo_labels(imgp, labp, CLASS_NAMES)
        ax.imshow(vis)
        ax.axis("off")
        ax.set_title(imgp.name)
    for ax in axes[len(sample_pairs):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 8. CLASS DISTRIBUTION
# ============================================================

def class_distribution(pairs):
    counts = {}
    for _, label in pairs:
        for cls, *_ in read_yolo_labels(label):
            counts[cls] = counts.get(cls, 0) + 1
    rows = []
    for cls in sorted(counts):
        rows.append({
            "class_id": cls,
            "class_name": CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else str(cls),
            "objects": counts[cls]
        })
    return pd.DataFrame(rows)

dist_df = class_distribution(pairs) if pairs else pd.DataFrame()
display(dist_df)

if not dist_df.empty:
    plt.figure(figsize=(14, 6))
    sns.barplot(data=dist_df, x="class_name", y="objects")
    plt.xticks(rotation=75, ha="right")
    plt.title("Bangladeshi Traffic Sign Class Distribution")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 9. CREATE LEAKAGE-SAFE TRAIN / VAL / TEST SPLIT
# ============================================================

def safe_split_pairs(pairs, train_ratio=0.8, val_ratio=0.1, seed=42):
    pairs = list(pairs)
    train_pairs, temp_pairs = train_test_split(
        pairs, test_size=(1-train_ratio), random_state=seed
    )
    relative_val = val_ratio / (1-train_ratio)
    val_pairs, test_pairs = train_test_split(
        temp_pairs, test_size=(1-relative_val), random_state=seed
    )
    return train_pairs, val_pairs, test_pairs

if pairs:
    train_pairs, val_pairs, test_pairs = safe_split_pairs(
        pairs, TRAIN_RATIO, VAL_RATIO, SEED
    )
    print("Train:", len(train_pairs))
    print("Val  :", len(val_pairs))
    print("Test :", len(test_pairs))
else:
    train_pairs, val_pairs, test_pairs = [], [], []

# IMPORTANT:
# If multiple images are frames/crops from the same original road sequence,
# replace this random split with a group/sequence-aware split before final results.


In [ ]:
# ============================================================
# 10. COPY DATA INTO YOLO DIRECTORY STRUCTURE
# ============================================================

def copy_pairs(pairs, split_name, root):
    img_out = Path(root) / split_name / "images"
    lab_out = Path(root) / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lab_out.mkdir(parents=True, exist_ok=True)

    for img, lab in tqdm(pairs, desc=f"Copying {split_name}"):
        # Keep names unique in case source folders contain duplicate filenames.
        target_img = img_out / img.name
        target_lab = lab_out / (img.stem + ".txt")

        if target_img.exists():
            target_img = img_out / f"{img.parent.name}_{img.name}"
            target_lab = lab_out / f"{img.parent.name}_{img.stem}.txt"

        shutil.copy2(img, target_img)
        shutil.copy2(lab, target_lab)

if pairs:
    if YOLO_DATASET.exists():
        shutil.rmtree(YOLO_DATASET)
    for split_name, split_pairs in [
        ("train", train_pairs),
        ("val", val_pairs),
        ("test", test_pairs)
    ]:
        copy_pairs(split_pairs, split_name, YOLO_DATASET)

    print("Prepared YOLO dataset at:", YOLO_DATASET)


In [ ]:
# ============================================================
# 11. WRITE DATASET YAML
# ============================================================

DATA_YAML = YOLO_DATASET / "data.yaml"

dataset_yaml = {
    "path": str(YOLO_DATASET),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES
}

DATA_YAML.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False))
print(DATA_YAML.read_text())


In [ ]:
# ============================================================
# 12. BASELINE YOLO TRAINING
# ============================================================

# The proposal asks for a clean-image YOLO baseline.
# We use YOLO11n for the lightweight baseline.

baseline_model = YOLO(YOLO_BASELINE)

baseline_results = baseline_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_BASELINE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_ROOT),
    name="baseline_yolo11n_clean",
    pretrained=True,
    seed=SEED,
    deterministic=True,
    patience=15,
    plots=True,
    verbose=True
)

BASELINE_WEIGHTS = RUNS_ROOT / "baseline_yolo11n_clean" / "weights" / "best.pt"
print("Baseline weights:", BASELINE_WEIGHTS)


In [ ]:
# ============================================================
# 13. BASELINE CLEAN TEST EVALUATION
# ============================================================

baseline_model = YOLO(str(BASELINE_WEIGHTS))

clean_val = baseline_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    verbose=True
)

def extract_yolo_metrics(metrics):
    box = metrics.box
    out = {
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
        "precision": float(box.mp),
        "recall": float(box.mr),
    }
    p = out["precision"]
    r = out["recall"]
    out["F1"] = 2*p*r/(p+r+1e-12)
    return out

baseline_clean_metrics = extract_yolo_metrics(clean_val)
baseline_clean_metrics


In [ ]:
# ============================================================
# 14. CORRUPTION FUNCTIONS — BD-TrafficSign-C
# ============================================================

def gaussian_blur(img, severity=1):
    k = [3, 5, 9][min(severity-1, 2)]
    return cv2.GaussianBlur(img, (k,k), 0)

def motion_blur(img, severity=1):
    k = [5, 9, 15][min(severity-1, 2)]
    kernel = np.zeros((k,k))
    kernel[k//2, :] = 1.0 / k
    return cv2.filter2D(img, -1, kernel)

def low_brightness(img, severity=1):
    factor = [0.65, 0.45, 0.30][min(severity-1, 2)]
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

def glare(img, severity=1):
    out = img.astype(np.float32).copy()
    H, W = img.shape[:2]
    radius = max(20, int(min(H,W) * [0.12,0.20,0.30][min(severity-1,2)]))
    cx = random.randint(0, W-1)
    cy = random.randint(0, H-1)
    overlay = np.zeros_like(out)
    cv2.circle(overlay, (cx,cy), radius, (255,255,255), -1)
    overlay = cv2.GaussianBlur(overlay, (0,0), radius/2)
    alpha = [0.25,0.45,0.65][min(severity-1,2)]
    out = cv2.addWeighted(out, 1-alpha, overlay, alpha, 0)
    return np.clip(out,0,255).astype(np.uint8)

def fog(img, severity=1):
    H, W = img.shape[:2]
    layer = np.full_like(img, 255, dtype=np.uint8)
    noise = np.random.normal(0, 18, (H,W)).astype(np.float32)
    noise = cv2.GaussianBlur(noise, (0,0), sigmaX=max(10, W/12))
    alpha = [0.15,0.30,0.45][min(severity-1,2)]
    foggy = img.astype(np.float32) * (1-alpha) + layer.astype(np.float32) * alpha
    foggy += noise[...,None] * 0.15
    return np.clip(foggy,0,255).astype(np.uint8)

def rain(img, severity=1):
    out = img.copy()
    H, W = img.shape[:2]
    n = int(H * W / 1200 * severity)
    for _ in range(n):
        x = random.randint(0,W-1)
        y = random.randint(0,H-1)
        length = random.randint(5, 20)
        thickness = 1 if severity < 3 else 2
        cv2.line(out, (x,y), (x-3,y+length), (210,210,210), thickness)
    return cv2.GaussianBlur(out, (3,3), 0.5)

def occlusion(img, severity=1):
    out = img.copy()
    H, W = img.shape[:2]
    frac = [0.08,0.16,0.25][min(severity-1,2)]
    rw = max(5, int(W * frac))
    rh = max(5, int(H * frac))
    x1 = random.randint(0, max(0,W-rw))
    y1 = random.randint(0, max(0,H-rh))
    cv2.rectangle(out, (x1,y1), (x1+rw,y1+rh), (25,25,25), -1)
    return out

def jpeg_compression(img, severity=1):
    quality = [45, 25, 10][min(severity-1,2)]
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        return img
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def small_object_resolution(img, severity=1):
    # Downsample then upsample. This keeps the normalized bounding boxes valid
    # while removing fine detail and simulating distant/small-sign imagery.
    H, W = img.shape[:2]
    scale = [0.70, 0.50, 0.30][min(severity-1,2)]
    small = cv2.resize(img, (max(8,int(W*scale)), max(8,int(H*scale))),
                       interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (W,H), interpolation=cv2.INTER_LINEAR)

CORRUPTIONS = {
    "gaussian_blur": gaussian_blur,
    "motion_blur": motion_blur,
    "low_brightness": low_brightness,
    "glare": glare,
    "fog": fog,
    "rain": rain,
    "occlusion": occlusion,
    "jpeg_compression": jpeg_compression,
    "small_object": small_object_resolution,
}

print("Corruptions:", list(CORRUPTIONS))


In [ ]:
# ============================================================
# 15. GENERATE CORRUPTED TEST SETS
# ============================================================

def generate_corrupted_split(clean_root, output_root, corruption_name, severity=2):
    clean_root = Path(clean_root)
    output_root = Path(output_root)
    fn = CORRUPTIONS[corruption_name]

    out_img = output_root / corruption_name / "images"
    out_lab = output_root / corruption_name / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lab.mkdir(parents=True, exist_ok=True)

    source_img_dir = clean_root / "test" / "images"
    source_lab_dir = clean_root / "test" / "labels"

    image_files = [p for p in source_img_dir.iterdir()
                   if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(image_files, desc=corruption_name):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        corrupted = fn(img, severity)
        cv2.imwrite(str(out_img / img_path.name), corrupted)

        lab = source_lab_dir / f"{img_path.stem}.txt"
        if lab.exists():
            shutil.copy2(lab, out_lab / lab.name)

for cname in CORRUPTIONS:
    generate_corrupted_split(YOLO_DATASET, CORRUPTION_ROOT, cname, severity=2)

print("Corrupted benchmark created at:", CORRUPTION_ROOT)


In [ ]:
# ============================================================
# 16. VISUALIZE THE CORRUPTION BENCHMARK
# ============================================================

if test_pairs:
    sample_img, sample_lab = random.choice(test_pairs)
    original = cv2.imread(str(sample_img))
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()

    axes[0].imshow(original)
    axes[0].set_title("clean")
    axes[0].axis("off")

    for ax, cname in zip(axes[1:], CORRUPTIONS):
        p = CORRUPTION_ROOT / cname / "images" / sample_img.name
        if p.exists():
            im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            ax.imshow(im)
            ax.set_title(cname)
        ax.axis("off")

    for ax in axes[len(CORRUPTIONS)+1:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 17. EVALUATE BASELINE ON ALL CORRUPTED TEST SETS
# ============================================================

def make_temp_yaml_for_image_dir(image_dir, names):
    temp_dir = Path(image_dir).parent.parent / "_eval_yaml"
    temp_dir.mkdir(parents=True, exist_ok=True)

    # YOLO val() expects labels beside the image directory in the standard layout.
    # Our corruption folders already have images/ and labels/.
    yaml_path = temp_dir / "data.yaml"
    d = {
        "path": str(Path(image_dir).parent.parent),
        "train": "images",
        "val": "images",
        "test": "images",
        "nc": len(names),
        "names": names
    }
    yaml_path.write_text(yaml.safe_dump(d, sort_keys=False))
    return yaml_path

corruption_results = []

# Evaluate clean test first.
corruption_results.append({
    "model": "baseline_yolo11n",
    "condition": "clean",
    **baseline_clean_metrics
})

for cname in CORRUPTIONS:
    eval_yaml = make_temp_yaml_for_image_dir(
        CORRUPTION_ROOT / cname / "images", CLASS_NAMES
    )

    m = baseline_model.val(
        data=str(eval_yaml),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        verbose=False
    )

    row = extract_yolo_metrics(m)
    row.update({
        "model": "baseline_yolo11n",
        "condition": cname
    })
    corruption_results.append(row)

baseline_robustness_df = pd.DataFrame(corruption_results)
baseline_robustness_df


In [ ]:
# ============================================================
# 18. ROBUSTNESS DROP ANALYSIS
# ============================================================

clean_map50 = float(
    baseline_robustness_df.loc[
        baseline_robustness_df.condition=="clean", "mAP50"
    ].iloc[0]
)

baseline_robustness_df["robustness_drop"] = clean_map50 - baseline_robustness_df["mAP50"]
baseline_robustness_df["relative_drop_%"] = (
    baseline_robustness_df["robustness_drop"] / max(clean_map50,1e-9) * 100
)

display(baseline_robustness_df)

plt.figure(figsize=(14,6))
plot_df = baseline_robustness_df[baseline_robustness_df.condition!="clean"]
sns.barplot(data=plot_df, x="condition", y="mAP50")
plt.axhline(clean_map50, linestyle="--", label=f"Clean mAP50={clean_map50:.3f}")
plt.xticks(rotation=60, ha="right")
plt.title("Baseline Robustness Across BD-TrafficSign-C")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,6))
sns.barplot(data=plot_df, x="condition", y="robustness_drop")
plt.xticks(rotation=60, ha="right")
plt.title("Robustness Drop = Clean mAP50 - Corrupted mAP50")
plt.tight_layout()
plt.show()


## Robust YOLO training

The proposal explicitly requires a comparison between standard/clean training and degradation-aware training. For the robust model we use YOLO's built-in augmentation controls for geometric/color variation and add the corruption family through the training configuration.

The exact augmentation controls can vary slightly by Ultralytics version, so the cell below keeps the experiment reproducible by recording all settings.

In [ ]:
# ============================================================
# 19. ROBUST YOLO TRAINING
# ============================================================

robust_model = YOLO(YOLO_BASELINE)

robust_results = robust_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_ROBUST,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_ROOT),
    name="robust_yolo11n_degradation_aware",
    pretrained=True,
    seed=SEED,
    deterministic=True,
    patience=15,
    plots=True,

    # Standard + stronger augmentation
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.45,
    degrees=10.0,
    translate=0.10,
    scale=0.50,
    shear=3.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.10,

    # Small-object awareness
    close_mosaic=10,

    verbose=True
)

ROBUST_WEIGHTS = RUNS_ROOT / "robust_yolo11n_degradation_aware" / "weights" / "best.pt"
print("Robust weights:", ROBUST_WEIGHTS)


### Important methodological note

The controlled corruption benchmark is the main robustness experiment. It should remain **test-only** when measuring the clean baseline.

For the strongest version of the research experiment, create a second training copy containing degradation-aware images and train a model on that copy. The next cell provides a reproducible way to build such a training set by applying randomly selected degradations to a fraction of the training images while preserving YOLO labels.

In [ ]:
# ============================================================
# 20. BUILD EXPLICIT DEGRADATION-AWARE TRAINING DATA
# ============================================================

ROBUST_DATASET = DATA_ROOT / "yolo_dataset_robust"

def build_robust_training_set(clean_dataset, robust_dataset,
                              corruption_probability=0.65,
                              severity=2):
    clean_dataset = Path(clean_dataset)
    robust_dataset = Path(robust_dataset)

    if robust_dataset.exists():
        shutil.rmtree(robust_dataset)

    # Copy validation and test unchanged.
    for split in ["val", "test"]:
        for sub in ["images", "labels"]:
            src = clean_dataset / split / sub
            dst = robust_dataset / split / sub
            dst.mkdir(parents=True, exist_ok=True)
            for p in src.iterdir():
                shutil.copy2(p, dst / p.name)

    train_img_src = clean_dataset / "train" / "images"
    train_lab_src = clean_dataset / "train" / "labels"

    train_img_dst = robust_dataset / "train" / "images"
    train_lab_dst = robust_dataset / "train" / "labels"
    train_img_dst.mkdir(parents=True, exist_ok=True)
    train_lab_dst.mkdir(parents=True, exist_ok=True)

    rng = random.Random(SEED)

    train_images = [p for p in train_img_src.iterdir()
                    if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(train_images, desc="Building robust training set"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        if rng.random() < corruption_probability:
            cname = rng.choice(list(CORRUPTIONS.keys()))
            img = CORRUPTIONS[cname](img, severity)

        cv2.imwrite(str(train_img_dst / img_path.name), img)

        lab = train_lab_src / f"{img_path.stem}.txt"
        if lab.exists():
            shutil.copy2(lab, train_lab_dst / lab.name)

    robust_yaml = robust_dataset / "data.yaml"
    robust_yaml.write_text(yaml.safe_dump({
        "path": str(robust_dataset),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES
    }, sort_keys=False))

    return robust_yaml

if pairs:
    ROBUST_DATA_YAML = build_robust_training_set(
        YOLO_DATASET,
        ROBUST_DATASET,
        corruption_probability=0.65,
        severity=2
    )
    print(ROBUST_DATA_YAML)


In [ ]:
# ============================================================
# 21. TRAIN THE EXPLICITLY DEGRADATION-AWARE MODEL
# ============================================================

# This is the strongest implementation of the proposal's
# "standard training vs degradation-aware training" comparison.

if pairs:
    explicit_robust_model = YOLO(YOLO_BASELINE)

    explicit_robust_model.train(
        data=str(ROBUST_DATA_YAML),
        epochs=EPOCHS_ROBUST,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUNS_ROOT),
        name="robust_yolo11n_explicit_corruptions",
        pretrained=True,
        seed=SEED,
        deterministic=True,
        patience=15,
        plots=True,
        verbose=True
    )

    EXPLICIT_ROBUST_WEIGHTS = (
        RUNS_ROOT / "robust_yolo11n_explicit_corruptions" / "weights" / "best.pt"
    )
    print("Explicit robust weights:", EXPLICIT_ROBUST_WEIGHTS)


In [ ]:
# ============================================================
# 22. EVALUATE ROBUST MODEL ON CLEAN + CORRUPTED TEST SETS
# ============================================================

def evaluate_model_on_benchmark(model_path, model_name):
    model = YOLO(str(model_path))
    rows = []

    # Clean
    m = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        verbose=False
    )
    rows.append({
        "model": model_name,
        "condition": "clean",
        **extract_yolo_metrics(m)
    })

    # Corruptions
    for cname in CORRUPTIONS:
        eval_yaml = make_temp_yaml_for_image_dir(
            CORRUPTION_ROOT / cname / "images", CLASS_NAMES
        )
        m = model.val(
            data=str(eval_yaml),
            split="test",
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            verbose=False
        )
        rows.append({
            "model": model_name,
            "condition": cname,
            **extract_yolo_metrics(m)
        })

    df = pd.DataFrame(rows)
    clean = df.loc[df.condition=="clean", "mAP50"].iloc[0]
    df["robustness_drop"] = clean - df["mAP50"]
    df["relative_drop_%"] = df["robustness_drop"] / max(clean,1e-9) * 100
    return df

robust_results_df = evaluate_model_on_benchmark(
    ROBUST_WEIGHTS, "robust_yolo11n_builtin_aug"
)

display(robust_results_df)

if pairs and Path(EXPLICIT_ROBUST_WEIGHTS).exists():
    explicit_results_df = evaluate_model_on_benchmark(
        EXPLICIT_ROBUST_WEIGHTS, "robust_yolo11n_explicit_corruptions"
    )
    display(explicit_results_df)
else:
    explicit_results_df = pd.DataFrame()


In [ ]:
# ============================================================
# 23. BASELINE VS ROBUST COMPARISON
# ============================================================

comparison_frames = [baseline_robustness_df, robust_results_df]
if not explicit_results_df.empty:
    comparison_frames.append(explicit_results_df)

all_results = pd.concat(comparison_frames, ignore_index=True)

summary = (
    all_results.groupby("model")
    .agg(
        clean_mAP50=("mAP50", lambda s: s.iloc[0]),
        mean_corrupted_mAP50=("mAP50", lambda s: s.iloc[1:].mean()),
        worst_corrupted_mAP50=("mAP50", lambda s: s.iloc[1:].min()),
        mean_robustness_drop=("robustness_drop", lambda s: s.iloc[1:].mean()),
        clean_precision=("precision", lambda s: s.iloc[0]),
        clean_recall=("recall", lambda s: s.iloc[0]),
        clean_F1=("F1", lambda s: s.iloc[0]),
    )
    .reset_index()
)

summary["robustness_improvement_vs_clean"] = (
    summary["clean_mAP50"] - summary["mean_corrupted_mAP50"]
)

display(summary)

plt.figure(figsize=(14,7))
sns.barplot(data=all_results, x="condition", y="mAP50", hue="model")
plt.xticks(rotation=60, ha="right")
plt.title("BD-TrafficGuard: Clean vs Corrupted mAP@0.5")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 24. CLASS-WISE PERFORMANCE
# ============================================================

# Ultralytics exposes per-class metrics through results.box.ap50 / ap.
# This cell computes class-wise AP50 on the clean test set.

def classwise_ap50(model_path):
    model = YOLO(str(model_path))
    m = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        verbose=False
    )

    ap50 = np.array(m.box.ap50).reshape(-1)
    rows = []
    for i, value in enumerate(ap50):
        rows.append({
            "class_id": i,
            "class_name": CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i),
            "AP50": float(value)
        })
    return pd.DataFrame(rows)

baseline_class_df = classwise_ap50(BASELINE_WEIGHTS)
display(baseline_class_df)

plt.figure(figsize=(14,6))
sns.barplot(data=baseline_class_df, x="class_name", y="AP50")
plt.xticks(rotation=75, ha="right")
plt.ylim(0,1)
plt.title("Class-wise AP@0.5 — Baseline")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 25. SPEED + MODEL SIZE
# ============================================================

def model_size_mb(path):
    return Path(path).stat().st_size / (1024**2)

def benchmark_inference(model_path, image_paths, imgsz=640, n=100):
    model = YOLO(str(model_path))
    image_paths = list(image_paths)[:n]

    if not image_paths:
        return {"avg_ms": np.nan, "fps": np.nan}

    # Warm-up
    for p in image_paths[:min(5,len(image_paths))]:
        model.predict(source=str(p), imgsz=imgsz, device=DEVICE, verbose=False)

    times = []
    for p in tqdm(image_paths, desc="Timing inference"):
        t0 = time.perf_counter()
        model.predict(source=str(p), imgsz=imgsz, device=DEVICE, verbose=False)
        times.append(time.perf_counter() - t0)

    avg_s = np.mean(times)
    return {
        "avg_ms": avg_s * 1000,
        "fps": 1.0 / avg_s
    }

test_image_paths = [
    p for p in (YOLO_DATASET / "test" / "images").iterdir()
    if p.suffix.lower() in IMG_EXTS
] if (YOLO_DATASET / "test" / "images").exists() else []

deployment_rows = []

for name, weight in [
    ("baseline_yolo11n", BASELINE_WEIGHTS),
    ("robust_yolo11n_builtin_aug", ROBUST_WEIGHTS),
]:
    if Path(weight).exists():
        speed = benchmark_inference(weight, test_image_paths, IMG_SIZE, n=50)
        deployment_rows.append({
            "model": name,
            "model_size_MB": model_size_mb(weight),
            **speed
        })

deployment_df = pd.DataFrame(deployment_rows)
display(deployment_df)


In [ ]:
# ============================================================
# 26. ABLATION 1 — MODEL SIZE
# ============================================================

# Proposal: YOLO-nano vs YOLO-small.
# Run this after the main experiments. It is intentionally separate
# because YOLO-small is more expensive.

def train_model_variant(model_name, run_name, epochs=30):
    model = YOLO(model_name)
    model.train(
        data=str(DATA_YAML),
        epochs=epochs,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUNS_ROOT),
        name=run_name,
        pretrained=True,
        seed=SEED,
        deterministic=True,
        patience=10,
        plots=True,
        verbose=True
    )
    return RUNS_ROOT / run_name / "weights" / "best.pt"

# Uncomment to run the expensive YOLO-small experiment:
#
# YOLO_SMALL_WEIGHTS = train_model_variant(
#     YOLO_SMALL, "ablation_yolo11s", epochs=EPOCHS_BASELINE
# )
#
# small_model = YOLO(str(YOLO_SMALL_WEIGHTS))
# small_metrics = extract_yolo_metrics(
#     small_model.val(
#         data=str(DATA_YAML), split="test",
#         imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE
#     )
# )
# print(small_metrics)


In [ ]:
# ============================================================
# 27. ABLATION 2 — INPUT RESOLUTION
# ============================================================

def evaluate_at_resolution(model_path, resolutions=(416, 640, 960)):
    model = YOLO(str(model_path))
    rows = []

    for size in resolutions:
        m = model.val(
            data=str(DATA_YAML),
            split="test",
            imgsz=size,
            batch=max(1, BATCH_SIZE // 2) if size >= 960 else BATCH_SIZE,
            device=DEVICE,
            verbose=False
        )
        metric = extract_yolo_metrics(m)
        metric["imgsz"] = size
        rows.append(metric)

    return pd.DataFrame(rows)

resolution_ablation_df = evaluate_at_resolution(
    BASELINE_WEIGHTS,
    resolutions=(416, 640, 960)
)
display(resolution_ablation_df)

plt.figure(figsize=(8,5))
sns.lineplot(data=resolution_ablation_df, x="imgsz", y="mAP50", marker="o")
plt.title("Input Resolution Ablation")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 28. ABLATION 3 — AUGMENTATION LEVEL
# ============================================================

# For a complete paper-quality ablation, train:
#   A: no augmentation
#   B: standard augmentation
#   C: degradation-aware augmentation
#
# The proposal explicitly lists this comparison.
#
# Uncomment the block below when GPU budget allows.

# def train_augmentation_ablation(name, aug_kwargs):
#     model = YOLO(YOLO_BASELINE)
#     model.train(
#         data=str(DATA_YAML),
#         epochs=EPOCHS_BASELINE,
#         imgsz=IMG_SIZE,
#         batch=BATCH_SIZE,
#         device=DEVICE,
#         workers=WORKERS,
#         project=str(RUNS_ROOT),
#         name=f"ablation_{name}",
#         pretrained=True,
#         seed=SEED,
#         deterministic=True,
#         patience=10,
#         plots=True,
#         **aug_kwargs
#     )
#     return RUNS_ROOT / f"ablation_{name}" / "weights" / "best.pt"
#
# no_aug = train_augmentation_ablation(
#     "no_augmentation",
#     dict(mosaic=0.0, mixup=0.0, degrees=0.0, translate=0.0, scale=0.0,
#          shear=0.0, perspective=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0)
# )
#
# standard_aug = train_augmentation_ablation(
#     "standard_augmentation",
#     dict(mosaic=1.0, mixup=0.0, degrees=10.0, translate=0.1, scale=0.5,
#          shear=3.0, perspective=0.0005, fliplr=0.5,
#          hsv_h=0.015, hsv_s=0.7, hsv_v=0.4)
# )
#
# Then evaluate all three with evaluate_model_on_benchmark().


## Optional two-stage detector + classifier

The proposal lists Faster R-CNN and EfficientNet/ResNet as candidate baselines. The following section implements a practical two-stage architecture:

**Road image → YOLO detector → crop detected sign → EfficientNet-B0 classifier**

This is useful for demonstrating the distinction between detection and recognition and for comparing one-stage YOLO recognition against a two-stage pipeline.

In [ ]:
# ============================================================
# 29. OPTIONAL CROPPED-SIGN DATASET FOR EFFICIENTNET-B0
# ============================================================

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

CLASSIFIER_ROOT = DATA_ROOT / "classifier_crops"

def crop_gt_signs(split_pairs, output_root, class_names, pad=0.05):
    output_root = Path(output_root)
    for i, cname in enumerate(class_names):
        (output_root / cname).mkdir(parents=True, exist_ok=True)

    for img_path, label_path in tqdm(split_pairs, desc="Cropping GT signs"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]

        for j, (cls, xc, yc, w, h) in enumerate(read_yolo_labels(label_path)):
            x1 = max(0, int((xc-w/2-pad)*W))
            y1 = max(0, int((yc-h/2-pad)*H))
            x2 = min(W, int((xc+w/2+pad)*W))
            y2 = min(H, int((yc+h/2+pad)*H))

            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            name = f"{img_path.stem}_{j}.jpg"
            cv2.imwrite(str(output_root / class_names[cls] / name), crop)

# This can take several minutes.
#
# Uncomment:
#
# if pairs:
#     if CLASSIFIER_ROOT.exists():
#         shutil.rmtree(CLASSIFIER_ROOT)
#     crop_gt_signs(train_pairs, CLASSIFIER_ROOT / "train", CLASS_NAMES)
#     crop_gt_signs(val_pairs, CLASSIFIER_ROOT / "val", CLASS_NAMES)
#     crop_gt_signs(test_pairs, CLASSIFIER_ROOT / "test", CLASS_NAMES)


In [ ]:
# ============================================================
# 30. OPTIONAL EFFICIENTNET-B0 CLASSIFIER
# ============================================================

class FolderClassifierDataset(Dataset):
    def __init__(self, root, class_names, transform=None):
        self.root = Path(root)
        self.class_names = class_names
        self.transform = transform
        self.samples = []

        for cls_idx, cls_name in enumerate(class_names):
            folder = self.root / cls_name
            if not folder.exists():
                continue
            for p in folder.iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    self.samples.append((p, cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y

classifier_weights = EfficientNet_B0_Weights.DEFAULT

train_tfms = classifier_weights.transforms()
val_tfms = classifier_weights.transforms()

# Uncomment after creating classifier crops:
#
# clf_train = FolderClassifierDataset(
#     CLASSIFIER_ROOT / "train", CLASS_NAMES, train_tfms
# )
# clf_val = FolderClassifierDataset(
#     CLASSIFIER_ROOT / "val", CLASS_NAMES, val_tfms
# )
#
# train_loader = DataLoader(
#     clf_train, batch_size=32, shuffle=True,
#     num_workers=WORKERS, pin_memory=torch.cuda.is_available()
# )
# val_loader = DataLoader(
#     clf_val, batch_size=32, shuffle=False,
#     num_workers=WORKERS, pin_memory=torch.cuda.is_available()
# )
#
# clf_model = efficientnet_b0(weights=classifier_weights)
# clf_model.classifier[1] = torch.nn.Linear(
#     clf_model.classifier[1].in_features, len(CLASS_NAMES)
# )
# clf_model = clf_model.to(DEVICE if DEVICE != "cpu" else "cpu")
#
# criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(clf_model.parameters(), lr=1e-4, weight_decay=1e-4)
#
# for epoch in range(10):
#     clf_model.train()
#     train_correct = train_total = 0
#     for xb, yb in train_loader:
#         xb = xb.to(DEVICE)
#         yb = yb.to(DEVICE)
#         optimizer.zero_grad()
#         logits = clf_model(xb)
#         loss = criterion(logits, yb)
#         loss.backward()
#         optimizer.step()
#         train_correct += (logits.argmax(1) == yb).sum().item()
#         train_total += len(yb)
#
#     print(f"Epoch {epoch+1}: train_acc={train_correct/train_total:.4f}")


In [ ]:
# ============================================================
# 31. OPTIONAL FASTER R-CNN BASELINE
# ============================================================

# This implements the proposal's two-stage detector comparison.
# It is intentionally optional because it is substantially more
# memory/time intensive than YOLO.

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

# The full Faster R-CNN training loop requires a custom Dataset that returns:
#   image tensor
#   target = {"boxes": Tensor[N,4], "labels": Tensor[N]}
#
# A compact implementation is provided below.

class YOLODetectionDataset(Dataset):
    def __init__(self, split, root, img_size=640):
        self.img_dir = Path(root) / split / "images"
        self.lab_dir = Path(root) / split / "labels"
        self.images = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in IMG_EXTS
        ])
        self.img_size = img_size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        p = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        lab = self.lab_dir / f"{p.stem}.txt"
        boxes, labels = [], []

        for cls, xc, yc, w, h in read_yolo_labels(lab):
            x1 = (xc-w/2)*W
            y1 = (yc-h/2)*H
            x2 = (xc+w/2)*W
            y2 = (yc+h/2)*H
            if x2 > x1 and y2 > y1:
                boxes.append([x1,y1,x2,y2])
                labels.append(cls+1)  # 0 reserved for background

        img_t = transforms.ToTensor()(Image.fromarray(img))
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        return img_t, target

def detection_collate(batch):
    return tuple(zip(*batch))

# Uncomment to train Faster R-CNN:
#
# frcnn = fasterrcnn_resnet50_fpn(
#     weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
#     num_classes=len(CLASS_NAMES)+1
# )
# frcnn.to(DEVICE)
#
# frcnn_train_ds = YOLODetectionDataset("train", YOLO_DATASET)
# frcnn_loader = DataLoader(
#     frcnn_train_ds, batch_size=2, shuffle=True,
#     num_workers=WORKERS, collate_fn=detection_collate
# )
#
# optimizer = torch.optim.SGD(
#     [p for p in frcnn.parameters() if p.requires_grad],
#     lr=0.005, momentum=0.9, weight_decay=0.0005
# )
#
# for epoch in range(10):
#     frcnn.train()
#     losses = []
#     for images, targets in tqdm(frcnn_loader, desc=f"FRCNN epoch {epoch+1}"):
#         images = [x.to(DEVICE) for x in images]
#         targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
#         loss_dict = frcnn(images, targets)
#         loss = sum(loss_dict.values())
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         losses.append(loss.item())
#     print("loss:", np.mean(losses))


In [ ]:
# ============================================================
# 32. QUALITATIVE FAILURE ANALYSIS
# ============================================================

def show_predictions(model_path, image_paths, n=12, conf=0.25, title="Predictions"):
    model = YOLO(str(model_path))
    selected = random.sample(list(image_paths), min(n, len(image_paths)))

    cols = 3
    rows = math.ceil(len(selected)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
    axes = np.array(axes).reshape(-1)

    for ax, p in zip(axes, selected):
        result = model.predict(
            source=str(p),
            imgsz=IMG_SIZE,
            conf=conf,
            device=DEVICE,
            verbose=False
        )[0]

        plotted = result.plot()
        plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
        ax.imshow(plotted)
        ax.set_title(Path(p).name)
        ax.axis("off")

    for ax in axes[len(selected):]:
        ax.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

if test_image_paths:
    show_predictions(
        BASELINE_WEIGHTS,
        test_image_paths,
        n=9,
        conf=0.25,
        title="Baseline — Clean Test Examples"
    )


In [ ]:
# ============================================================
# 33. QUALITATIVE CORRUPTION FAILURE ANALYSIS
# ============================================================

def show_corrupted_predictions(model_path, corruption_name, n=9, conf=0.25):
    img_dir = CORRUPTION_ROOT / corruption_name / "images"
    paths = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    show_predictions(
        model_path, paths, n=n, conf=conf,
        title=f"{Path(model_path).stem} — {corruption_name}"
    )

for cname in ["gaussian_blur", "low_brightness", "occlusion", "small_object"]:
    if (CORRUPTION_ROOT / cname / "images").exists():
        show_corrupted_predictions(BASELINE_WEIGHTS, cname, n=6)


In [ ]:
# ============================================================
# 34. EXPORT RESULTS FOR REPORT / PAPER
# ============================================================

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_robustness_df.to_csv(
    RESULTS_DIR / "baseline_corruption_results.csv", index=False
)
robust_results_df.to_csv(
    RESULTS_DIR / "robust_corruption_results.csv", index=False
)
summary.to_csv(
    RESULTS_DIR / "model_summary.csv", index=False
)
deployment_df.to_csv(
    RESULTS_DIR / "deployment_metrics.csv", index=False
)
baseline_class_df.to_csv(
    RESULTS_DIR / "baseline_classwise_ap50.csv", index=False
)
resolution_ablation_df.to_csv(
    RESULTS_DIR / "resolution_ablation.csv", index=False
)

# Save experiment configuration
config = {
    "seed": SEED,
    "img_size": IMG_SIZE,
    "epochs_baseline": EPOCHS_BASELINE,
    "epochs_robust": EPOCHS_ROBUST,
    "batch_size": BATCH_SIZE,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "baseline_model": YOLO_BASELINE,
    "robust_model": YOLO_BASELINE,
    "corruptions": list(CORRUPTIONS.keys()),
    "classes": CLASS_NAMES,
}
(RESULTS_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2))

print("Results saved to:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(p.name)


In [ ]:
# ============================================================
# 35. FINAL PROJECT TABLE
# ============================================================

final_table = summary.copy()

# Make a cleaner report-style table.
final_table = final_table.rename(columns={
    "clean_mAP50": "Clean mAP@0.5",
    "mean_corrupted_mAP50": "Mean Corrupted mAP@0.5",
    "worst_corrupted_mAP50": "Worst Corrupted mAP@0.5",
    "mean_robustness_drop": "Mean Robustness Drop",
    "clean_precision": "Precision",
    "clean_recall": "Recall",
    "clean_F1": "F1"
})

display(final_table.round(4))


# Final checklist — BD-TrafficGuard

Before submitting the project, make sure the following are completed:

1. **Dataset**
   - Bangladesh Road Traffic Sign Dataset downloaded
   - 10,259 images / 31 classes verified against the dataset
   - annotation format verified
   - corrupted images checked
   - class distribution reported

2. **Data split**
   - train/validation/test created
   - no augmented test images
   - if images originate from sequences, use a sequence-aware split

3. **Baseline**
   - YOLO11n trained on clean images
   - clean test metrics recorded

4. **BD-TrafficSign-C**
   - Gaussian blur
   - motion blur
   - low brightness
   - glare
   - fog
   - rain
   - occlusion
   - JPEG compression
   - small-object/resolution degradation

5. **Robust model**
   - degradation-aware augmentation
   - clean test performance
   - every corrupted test-set performance

6. **Metrics**
   - mAP@0.5
   - mAP@0.5:0.95
   - precision
   - recall
   - F1
   - class-wise AP/recall
   - robustness drop
   - inference time
   - FPS
   - model size

7. **Ablations**
   - no augmentation vs standard vs degradation-aware
   - YOLO11n vs YOLO11s
   - 416 vs 640 vs 960
   - optional class-balanced sampling
   - optional YOLO vs Faster R-CNN / two-stage detector

8. **Qualitative analysis**
   - correct difficult detections
   - missed small signs
   - visually similar classes
   - false positives
   - blur/occlusion failures

9. **Deployment**
   - export model
   - measure inference speed
   - record model size
   - optionally export ONNX for edge/mobile experiments

The core research claim should be based on measured results, not assumed results: whether degradation-aware training reduces the clean-to-corrupted performance drop on Bangladeshi traffic signs.